In [1]:
from __future__ import annotations

import csv
import json
import subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch')
LOCAL_ROOT = Path('/content/charades_raw_from_scratch')

DOWNLOAD_DIR = DRIVE_ROOT / 'downloads'
DRIVE_ANN_DIR = DRIVE_ROOT / 'raw' / 'annotations'
MANIFEST_DIR = DRIVE_ROOT / 'manifests'
LOCAL_RAW_DIR = LOCAL_ROOT / 'raw'
LOCAL_ANN_DIR = LOCAL_RAW_DIR / 'annotations'
VIDEO_DIR = LOCAL_RAW_DIR / 'videos_480p'
for p in [DOWNLOAD_DIR, DRIVE_ANN_DIR, MANIFEST_DIR, LOCAL_RAW_DIR, LOCAL_ANN_DIR, VIDEO_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ANNOTATIONS_URL = 'https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip'
VIDEOS_URL = 'https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip'

def run(cmd):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)

def wget(url: str, out_path: Path):
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f'skip existing: {out_path.name}')
        return
    run(['wget', '-c', '-O', str(out_path), url])

print('DRIVE_ROOT =', DRIVE_ROOT)
print('LOCAL_ROOT =', LOCAL_ROOT)


Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch
LOCAL_ROOT = /content/charades_raw_from_scratch


In [2]:
annotations_zip = DOWNLOAD_DIR / 'Charades.zip'
videos_zip = DOWNLOAD_DIR / 'Charades_v1_480.zip'

wget(ANNOTATIONS_URL, annotations_zip)
wget(VIDEOS_URL, videos_zip)

run(['unzip', '-n', str(annotations_zip), '-d', str(DRIVE_ANN_DIR)])
run(['unzip', '-n', str(annotations_zip), '-d', str(LOCAL_ANN_DIR)])
run(['unzip', '-n', str(videos_zip), '-d', str(VIDEO_DIR)])

print('drive annotations root:', DRIVE_ANN_DIR)
print('local annotations root:', LOCAL_ANN_DIR)
print('videos root:', VIDEO_DIR)


$ wget -c -O /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/downloads/Charades.zip https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip
$ wget -c -O /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/downloads/Charades_v1_480.zip https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip
$ unzip -n /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/downloads/Charades.zip -d /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/raw/annotations
$ unzip -n /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/downloads/Charades.zip -d /content/charades_raw_from_scratch/raw/annotations
$ unzip -n /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/downloads/Charades_v1_480.zip -d /content/charades_raw_from_scratch/raw/videos_480p
drive annotations root: /content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch/raw/annotations
l

In [3]:
classes_path = LOCAL_ANN_DIR / 'Charades' / 'Charades_v1_classes.txt'
train_csv = LOCAL_ANN_DIR / 'Charades' / 'Charades_v1_train.csv'
test_csv = LOCAL_ANN_DIR / 'Charades' / 'Charades_v1_test.csv'

assert classes_path.exists(), classes_path
assert train_csv.exists(), train_csv
assert test_csv.exists(), test_csv

def load_class_map(path: Path):
    class_codes = []
    class_names = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            code, name = line.split(' ', 1)
            class_codes.append(code)
            class_names.append(name)
    class_to_idx = {code: i for i, code in enumerate(class_codes)}
    return class_to_idx, class_codes, class_names

CLASS_TO_IDX, IDX_TO_CODE, IDX_TO_NAME = load_class_map(classes_path)

def parse_row(row):
    actions_raw = row.get('actions', '')
    labels = []
    action_timings = []
    action_codes = []
    for item in actions_raw.split(';'):
        item = item.strip()
        if not item:
            continue
        code, start, end = item.split()
        labels.append(CLASS_TO_IDX[code])
        action_timings.append([float(start), float(end)])
        action_codes.append(code)
    descriptions = [x for x in row.get('descriptions', '').split(';') if x]
    objects = [x for x in row.get('objects', '').split(';') if x]
    return {
        'video_id': row['id'],
        'subject': row.get('subject', ''),
        'scene': row.get('scene', ''),
        'quality': int(row['quality']) if row.get('quality') not in ('', None) else -100,
        'relevance': int(row['relevance']) if row.get('relevance') not in ('', None) else -100,
        'verified': row.get('verified', ''),
        'script': row.get('script', ''),
        'objects': objects,
        'descriptions': descriptions,
        'labels': labels,
        'action_codes': action_codes,
        'action_timings': action_timings,
        'length': float(row['length']) if row.get('length') not in ('', None) else float('nan'),
    }

def read_split(csv_path: Path, split_name: str):
    rows = []
    with csv_path.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            item = parse_row(row)
            item['split'] = split_name
            item['video_path'] = str((VIDEO_DIR / 'Charades_v1' / f"{item['video_id']}.mp4").resolve())
            item['num_actions'] = len(item['labels'])
            rows.append(item)
    return rows

train_rows = read_split(train_csv, 'train')
test_rows = read_split(test_csv, 'test')
all_rows = train_rows + test_rows

missing_videos = [r['video_id'] for r in all_rows if not Path(r['video_path']).exists()]
print('train rows:', len(train_rows))
print('test rows:', len(test_rows))
print('all rows:', len(all_rows))
print('missing videos:', len(missing_videos))
print('classes:', len(IDX_TO_CODE))
print('sample row:')
print(json.dumps(all_rows[0], indent=2)[:2000])


train rows: 7985
test rows: 1863
all rows: 9848
missing videos: 9848
classes: 157
sample row:
{
  "video_id": "46GP8",
  "subject": "HR43",
  "scene": "Kitchen",
  "quality": 6,
  "relevance": 7,
  "verified": "Yes",
  "script": "A person cooking on a stove while watching something out a window.",
  "objects": [
    "food",
    "stove",
    "window"
  ],
  "descriptions": [
    "A person cooks food on a stove before looking out of a window."
  ],
  "labels": [
    92,
    147
  ],
  "action_codes": [
    "c092",
    "c147"
  ],
  "action_timings": [
    [
      11.9,
      21.2
    ],
    [
      0.0,
      12.6
    ]
  ],
  "length": 24.83,
  "split": "train",
  "video_path": "/content/charades_raw_from_scratch/raw/videos_480p/Charades_v1/46GP8.mp4",
  "num_actions": 2
}


In [4]:
df = pd.DataFrame(all_rows)
train_df = df[df['split'] == 'train'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

# Derived validation split from official train split
rng = np.random.default_rng(42)
train_video_ids = train_df['video_id'].tolist()
rng.shuffle(train_video_ids)
val_n = max(1, int(round(0.1 * len(train_video_ids))))
val_ids = set(train_video_ids[:val_n])
train_ids = set(train_video_ids[val_n:])
train_sub_df = train_df[train_df['video_id'].isin(train_ids)].reset_index(drop=True)
val_df = train_df[train_df['video_id'].isin(val_ids)].reset_index(drop=True)

def class_frequency(frame: pd.DataFrame):
    counter = Counter()
    for labels in frame['labels']:
        counter.update(labels)
    return counter

def interval_stats(frame: pd.DataFrame):
    lengths = []
    num_intervals = []
    for timings in frame['action_timings']:
        num_intervals.append(len(timings))
        for start, end in timings:
            lengths.append(float(end) - float(start))
    return np.array(lengths, dtype=float), np.array(num_intervals, dtype=int)

action_lengths, action_counts = interval_stats(df)
class_freq = class_frequency(df)

survey = {
    'dataset_root': str(DRIVE_ROOT),
    'annotation_file_train': str(train_csv),
    'annotation_file_test': str(test_csv),
    'video_root': str(VIDEO_DIR),
    'num_train_videos': int(len(train_df)),
    'num_test_videos': int(len(test_df)),
    'num_total_videos': int(len(df)),
    'num_val_videos_derived': int(len(val_df)),
    'num_train_videos_derived': int(len(train_sub_df)),
    'num_classes': int(len(IDX_TO_CODE)),
    'avg_actions_per_video': float(df['num_actions'].mean()),
    'median_actions_per_video': float(df['num_actions'].median()),
    'avg_video_length_sec': float(df['length'].mean()),
    'median_video_length_sec': float(df['length'].median()),
    'avg_action_interval_sec': float(action_lengths.mean()),
    'median_action_interval_sec': float(np.median(action_lengths)),
    'train_split_class_count_top5': Counter(class_freq).most_common(5),
    'scene_counts_top5': Counter(df['scene']).most_common(5),
}

print(json.dumps(survey, indent=2))
print('\nTop 10 classes:')
top10 = Counter(class_freq).most_common(10)
for idx, cnt in top10:
    print(idx, IDX_TO_NAME[idx], cnt)

print('\nSplit counts:')
print('train official:', len(train_df))
print('test official :', len(test_df))
print('train derived :', len(train_sub_df))
print('val derived   :', len(val_df))

# Save artifacts
manifest_path = MANIFEST_DIR / 'charades_raw_manifest.json'
train_manifest_path = MANIFEST_DIR / 'charades_raw_train_manifest.json'
val_manifest_path = MANIFEST_DIR / 'charades_raw_val_manifest.json'
test_manifest_path = MANIFEST_DIR / 'charades_raw_test_manifest.json'
survey_path = MANIFEST_DIR / 'charades_raw_survey.json'
class_map_path = MANIFEST_DIR / 'charades_class_map.json'
class_names_path = MANIFEST_DIR / 'charades_class_names.txt'

with manifest_path.open('w', encoding='utf-8') as f:
    json.dump(all_rows, f, indent=2)
with train_manifest_path.open('w', encoding='utf-8') as f:
    json.dump(train_sub_df.to_dict(orient='records'), f, indent=2)
with val_manifest_path.open('w', encoding='utf-8') as f:
    json.dump(val_df.to_dict(orient='records'), f, indent=2)
with test_manifest_path.open('w', encoding='utf-8') as f:
    json.dump(test_df.to_dict(orient='records'), f, indent=2)
with survey_path.open('w', encoding='utf-8') as f:
    json.dump(survey, f, indent=2)
with class_map_path.open('w', encoding='utf-8') as f:
    json.dump({'class_to_idx': CLASS_TO_IDX, 'idx_to_code': IDX_TO_CODE, 'idx_to_name': IDX_TO_NAME}, f, indent=2)
with class_names_path.open('w', encoding='utf-8') as f:
    for i, name in enumerate(IDX_TO_NAME):
        f.write(f'{i:03d} {IDX_TO_CODE[i]} {name}\n')

print('saved manifest:', manifest_path)
print('saved survey  :', survey_path)


{
  "dataset_root": "/content/drive/MyDrive/momentlens_datasets/charades_raw_from_scratch",
  "annotation_file_train": "/content/charades_raw_from_scratch/raw/annotations/Charades/Charades_v1_train.csv",
  "annotation_file_test": "/content/charades_raw_from_scratch/raw/annotations/Charades/Charades_v1_test.csv",
  "video_root": "/content/charades_raw_from_scratch/raw/videos_480p",
  "num_train_videos": 7985,
  "num_test_videos": 1863,
  "num_total_videos": 9848,
  "num_val_videos_derived": 798,
  "num_train_videos_derived": 7187,
  "num_classes": 157,
  "avg_actions_per_video": 6.75264012997563,
  "median_actions_per_video": 6.0,
  "avg_video_length_sec": 29.663260560519905,
  "median_video_length_sec": 30.54,
  "avg_action_interval_sec": 12.757266766917294,
  "median_action_interval_sec": 8.8,
  "train_split_class_count_top5": [
    [
      97,
      1883
    ],
    [
      154,
      1722
    ],
    [
      59,
      1628
    ],
    [
      61,
      1442
    ],
    [
      106,
    